In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

prices = spark.table("workspace.default.silver_prices")
consumption = spark.table("workspace.default.silver_consumption")
weather = spark.table("workspace.default.silver_weather")

prices.select(F.min("start_utc"), F.max("start_utc")).show()
consumption.select(F.min("start_utc"), F.max("start_utc")).show()
weather.select(F.min("time_utc"), F.max("time_utc")).show()

+-------------------+-------------------+
|     min(start_utc)|     max(start_utc)|
+-------------------+-------------------+
|2026-10-04 22:00:00|2026-10-20 21:00:00|
+-------------------+-------------------+

+-------------------+-------------------+
|     min(start_utc)|     max(start_utc)|
+-------------------+-------------------+
|2026-10-04 22:00:00|2026-10-18 21:00:00|
+-------------------+-------------------+

+-------------------+-------------------+
|      min(time_utc)|      max(time_utc)|
+-------------------+-------------------+
|2026-10-09 18:00:00|2026-10-30 06:00:00|
+-------------------+-------------------+



In [0]:
cons_hour = (
    consumption
    .groupBy("area", "start_utc")
    .agg(F.sum("quantity_kwh").alias("consumption_kwh"))
)
print(cons_hour.count())

1680


In [0]:
weather_d1 = (
    weather
    .filter(F.col("forecast_date") == F.date_sub(F.to_date("time_utc"), 1))
    .select(
        "area",
        F.col("time_utc").alias("start_utc"),
        "temperature_c",
        "wind_speed_ms",
        "precip_1h_mm",
    )
)
print(weather_d1.count())

1440


In [0]:
gold = (
    prices
    .select("area", "start_utc", "price_nok_kwh")
    .join(cons_hour, ["area", "start_utc"], "inner")
    .join(weather_d1, ["area", "start_utc"], "inner")
)
print(gold.count())

1070


In [0]:
yesterday = prices.select(
    "area",
    (F.col("start_utc") + F.expr("INTERVAL 1 DAY")).alias("start_utc"),
    F.col("price_nok_kwh").alias("price_yesterday_nok_kwh"),
)

gold = gold.join(yesterday, ["area", "start_utc"], "left")
print(gold.count())

1070


In [0]:
gold.write.mode("overwrite").saveAsTable("workspace.default.gold_hourly")
spark.table("workspace.default.gold_hourly").orderBy("area", "start_utc").show(5, truncate=False)

+----+-------------------+-------------+------------------+-------------+-------------+------------+-----------------------+
|area|start_utc          |price_nok_kwh|consumption_kwh   |temperature_c|wind_speed_ms|precip_1h_mm|price_yesterday_nok_kwh|
+----+-------------------+-------------+------------------+-------------+-------------+------------+-----------------------+
|NO1 |2026-10-10 00:00:00|0.15432      |3179324.024       |7.2          |4.5          |0.0         |0.93676                |
|NO1 |2026-10-10 01:00:00|0.16172      |3131047.075       |6.9          |4.8          |0.1         |0.88974                |
|NO1 |2026-10-10 02:00:00|0.17683      |3099900.7800000003|6.7          |5.3          |0.4         |0.88181                |
|NO1 |2026-10-10 03:00:00|0.15261      |3094332.6199999996|6.6          |5.1          |1.2         |0.98068                |
|NO1 |2026-10-10 04:00:00|0.24617      |3151036.974       |6.1          |4.0          |1.5         |1.20714                |


In [0]:
(
    spark.table("workspace.default.gold_hourly")
    .withColumn("abs_error", F.abs(F.col("price_nok_kwh") - F.col("price_yesterday_nok_kwh")))
    .groupBy("area")
    .agg(
        F.round(F.avg("abs_error"), 3).alias("mae_nok_kwh"),
        F.round(F.avg("price_nok_kwh"), 3).alias("avg_price_nok_kwh"),
    )
    .orderBy("area")
    .show()
)

+----+-----------+-----------------+
|area|mae_nok_kwh|avg_price_nok_kwh|
+----+-----------+-----------------+
| NO1|      0.234|            1.468|
| NO2|      0.274|            1.531|
| NO3|      0.277|             1.13|
| NO4|      0.034|            0.249|
| NO5|      0.203|             1.42|
+----+-----------+-----------------+

